## Step2-Process-CMIP

Takes CMIP surface runoff and sums over ISMIP7 drainage basins to generate monthly 1850-2299 per-basin subglacial discharge from CMIP. There is code to make a number of plots along the way that may be useful in sense-checking or debugging; this code is commented by default.

Creator: Donald Slater, donald.slater@ed.ac.uk, 27 June 2025.

Cleaned up by Donald Slater, 9 Feb 2026.

#### Required files/inputs
- Drainage basins definition file *ismip7_runoff_basins_nearestbelowsl.nc*
- CMIP-derived runoff files. For CESM2-WACCM, these are of the form *runoff_CESM2-WACCM_ssp585_SDBN1_1850.nc* etc.

#### Outputs
- Monthly ISMIP7 per basin subglacial discharge, in chunks as defined by start_years and end_years variables in the setup section. For CESM2-WACCM, these are of the form *Q_CESM2-WACCM_ssp585_SDBN1_startyear_endyear.nc*.

### Setup

In [1]:
# location of file that contains the drainage basin definition
basins_file = 'ismip7_runoff_basins_nearestbelowsl.nc'

# nominal number of basins outside of continental shelf
n_nominal_basins = 50

# CMIP runoff directories - this is a little awkward given the historical and projection are in different directories
runoff_dirs = list() # list of directories for CMIP runoff files
start_years = list() # list of start years for each directory
final_years = list() # list of end years for each directory
# historical: 1850-2014
runoff_dirs.append('/Users/dslater2/Documents/CESM2-WACCM/historical/runoff/')
start_years.append(1850)
final_years.append(2014)
# projection: 2015-2100
runoff_dirs.append('/Users/dslater2/Documents/CESM2-WACCM/ssp585/runoff/projection/')
start_years.append(2015)
final_years.append(2100)
# extension: 2101-2299 (N.B. CESM-WACCM only goes to 2299, other CMIP models may go to 2300)
# another N.B. on globus, projection and extension are not separated and they don't need to be here either
# it just breaks up the processing
runoff_dirs.append('/Users/dslater2/Documents/CESM2-WACCM/ssp585/runoff/extension/')
start_years.append(2101)
final_years.append(2299)

# output files go into the same directories as runoff_dirs

### Imports

In [2]:
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import pandas as pd
import xarray as xr
import cftime

#### Define function that does the main processing

In [14]:
def process_runoff(runoff_dir,basins_file,n_nominal_basins,start_year,final_year):

    # get CMIP runoff filenames
    print(runoff_dir)
    runoff_files = sorted(glob.glob(os.path.join(runoff_dir, "runoff*.nc")))
    
    # throw out the CMIP filenames that are outside of the timespan start_year-final_year
    remove_idxs = list()
    for ifile, runoff_file in enumerate(runoff_files):
        year = int(runoff_file.split('_')[4].split('.')[0])
        if year < start_year or year > final_year:
            remove_idxs.append(ifile)
    runoff_files = [value for index, value in enumerate(runoff_files) if index not in remove_idxs]
    
    # output file (works at least for my directory structure and CESM2 file names)
    firstyear = runoff_files[0][-7:-3]
    lastyear = runoff_files[-1][-7:-3]
    output_file = runoff_files[0].split('/')[-1].replace('runoff', "Q").replace("monthly_", "").replace(firstyear, firstyear + "_" + lastyear)
    output_file = runoff_dir + output_file
    print(output_file)
    # return

    # delete existing output file if it exists
    if os.path.exists(output_file):
        os.remove(output_file)

    # load drainage basins
    basins = Dataset(basins_file, mode='r')
    x = basins.variables['x'][:].filled(np.nan)
    y = basins.variables['y'][:].filled(np.nan)
    b = basins.variables['basin'][:].filled(np.nan)
    m = basins.variables['icesheetmask'][:].filled(np.nan)
    o = basins.variables['convexhullmask'][:].filled(np.nan)

    # get spatial indices corresponding to each basin (used later)
    b_unique = np.unique(b[:]) # get unique basin ids
    basin_indices = {}
    runoff_indices = {}
    for i in b_unique:
        basin_indices[i] = np.where(b==i) # indices of extended basins
        runoff_indices[i] = np.where((b==i) & (m==1)) # indices giving where to sum runoff (excludes ice caps)

    # indices outside convex hull
    outside_indices = np.where(o==1)

    # create output netcdf
    with Dataset(output_file,'w',format='NETCDF4') as nc_out:
        
        # dimensions
        nc_out.createDimension('x', len(x))
        nc_out.createDimension('y', len(y))
        nc_out.createDimension('time', None) # unlimited time dimension

        # variables
        x_var = nc_out.createVariable('x', np.int32, ('x',)) # int32 sufficient for coords
        y_var = nc_out.createVariable('y', np.int32, ('y',))
        time_var = nc_out.createVariable('time', np.float32, ('time',))
        Q_var = nc_out.createVariable('Q', np.float32, ('time', 'y', 'x'),zlib=True, complevel=9)

        # metadata
        x_var.standard_name = 'projection_x_coordinate'
        x_var.units = 'meters'
        x_var.long_name = 'x coordinate in EPSG:3413 projection'
        x_var.axis = 'X'

        y_var.standard_name = 'projection_x_coordinate'
        y_var.units = 'meters'
        y_var.long_name = 'y coordinate in EPSG:3413 projection'
        y_var.axis = 'Y'

        # NB this must be the same as the input runoff files
        time_var.standard_name = 'time'
        time_var.units = 'months since 1850-01-15 00:00:00'
        time_var.calendar = 'noleap'
        time_var.long_name = 'time'

        Q_var.standard_name = 'subglacial_discharge_volume_flux'
        Q_var.units = 'm3 s-1'
        Q_var.long_name = 'basin subglacial discharge'
        Q_var.grid_mapping = 'crs'

        # write coordinates
        x_var[:] = np.array(x, dtype=np.int32)
        y_var[:] = np.array(y, dtype=np.int32)

        # Create grid mapping variable with EPSG:3413 info
        crs_var = nc_out.createVariable('crs', 'i4')
        crs_var.grid_mapping_name = 'polar_stereographic'
        crs_var.straight_vertical_longitude_from_pole = -45.0
        crs_var.latitude_of_projection_origin = 90.0
        crs_var.standard_parallel = 70.0
        crs_var.false_easting = 0.0
        crs_var.false_northing = 0.0
        crs_var.semi_major_axis = 6378137.0
        crs_var.inverse_flattening = 298.257223563

        # Global attributes
        nc_out.title = 'Monthly Basin Subglacial Discharge from CESM2-WACCM, processed by Donald Slater'
        nc_out.history = 'Created ' + str(np.datetime64('now'))

    # Initialize time index for appending
    time_index = 0

    # loop over runoff files
    for file_path in runoff_files:
        print(f"Processing {file_path}")

        with Dataset(file_path, 'r') as runoff:
            
            # read data for this year
            xrunoff = runoff.variables['x'][:].filled(np.nan)
            yrunoff = runoff.variables['y'][:].filled(np.nan)
            t = runoff.variables['time'][:].filled(np.nan)
            r = runoff.variables['runoff'][:].filled(np.nan)

            # check coords match with basins
            if (np.sum(xrunoff-x)!=0) or (np.sum(yrunoff-y)!=0):
                print('Warning: runoff coordinates differ from basins')

            # convert runoff units mmwe/month to m3/s
            r = 1000 * 1000 * r / (1000 * (365 / 12) * 86400)

            # initialize output Q array for this file
            Q = np.full_like(r, np.nan)

            # sum runoff over basins
            for i in b_unique: # loop over drainage basins
                inds = basin_indices[i] # obtain pixels in drainage basin i 
                inds_runoff = runoff_indices[i]
                for k in range(r.shape[0]): # loop over months
                    Q[k,inds[0],inds[1]] = np.sum(r[k,inds_runoff[0],inds_runoff[1]]) # assign subglacial discharge

            # assign outside of convex hull values to nominal value (which is total runoff split over 50 basins)
            for k in range(r.shape[0]):
                Q[k,outside_indices[0],outside_indices[1]] = np.sum(np.unique(Q[k,:,:]))/n_nominal_basins

        # append data to output netcdf
        with Dataset(output_file, 'a') as nc_out:
            nc_out.variables['time'][time_index:time_index + len(t)] = t.astype(np.float32)
            nc_out.variables['Q'][time_index:time_index + len(t), :, :] = Q.astype(np.float32)

        time_index += len(t)

    print("All files processed and appended.")

#### Do the processing (takes nearly an hour for full 1850-2300)

In [ ]:
for (runoff_dir, start_year, final_year) in zip(runoff_dirs, start_years, final_years):
    print('Processing files in ' + runoff_dir)
    process_runoff(runoff_dir,basins_file,n_nominal_basins,start_year,final_year)
    print('')

### Old code below here

#### Get ISMIP7 drainage basins that have been pre-generated in matlab scripts

In [ ]:
# # load drainage basins
# basins = Dataset(basins_file, mode='r')
# x = basins.variables['x'][:].filled(np.nan)
# y = basins.variables['y'][:].filled(np.nan)
# b = basins.variables['basin'][:].filled(np.nan)
# m = basins.variables['icesheetmask'][:].filled(np.nan)
# o = basins.variables['convexhullmask'][:].filled(np.nan)

# # get spatial indices corresponding to each basin (used later)
# b_unique = np.unique(b[:]) # get unique basin ids
# basin_indices = {}
# runoff_indices = {}
# for i in b_unique:
#     basin_indices[i] = np.where(b==i) # indices of extended basins
#     runoff_indices[i] = np.where((b==i) & (m==1)) # indices giving where to sum runoff (excludes ice caps)

# # indices outside convex hull
# outside_indices = np.where(o==1)

# # plot drainage basins and mask
# plt.figure(figsize=(10, 5))
# plt.subplot(131)
# plt.pcolormesh(x, y, m, cmap='gray')
# plt.axis('equal')
# plt.title('Ice Sheet Mask')
# plt.colorbar(label='mask value')
# plt.subplot(132)
# plt.pcolormesh(x, y, o, cmap='gray')
# plt.axis('equal')
# plt.title('Convex Hull Mask')
# plt.colorbar(label='mask value')
# plt.subplot(133)
# plt.title('Drainage Basins')
# plt.pcolormesh(x, y, b)
# plt.axis('equal')
# plt.colorbar(label='basin id')
# plt.tight_layout()
# plt.show()

#### Read runoff and estimate subglacial discharge for single file to check process

Estimate subglacial discharge for a given basin by summing CMIP surface runoff over that basin.

In [ ]:
# # location of single test file
# cesm_runoff_file = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/PROCESSEDmon/runoff/SDBN1/runoff_CESM2-WACCM_ssp585_SDBN1_2220.nc'

# # load runoff
# runoff = Dataset(cesm_runoff_file, mode='r')
# xrunoff = runoff.variables['x'][:].filled(np.nan)
# yrunoff = runoff.variables['y'][:].filled(np.nan)
# time_var = runoff.variables['time'][:].filled(np.nan)
# r = runoff.variables['runoff'][:].filled(np.nan)

# # check coordinate system for runoff is the same as for the basins
# if (np.sum(xrunoff-x)!=0) | (np.sum(yrunoff-y)!=0):
#     print('Warning: runoff coordinates not the same as basin coordinates')
    
# # convert runoff units from mmwe/month to m3/s (assumes 365/12 days in a month)
# r = 1000*1000*r/(1000*(365/12)*86400)

# # sum runoff values over drainage basins to obtain subglacial discharge
# Q = np.nan*r # initialise subglacial discharge field
# for i in b_unique: # loop over drainage basins
#     inds = basin_indices[i]
#     inds_runoff = runoff_indices[i]  
#     for k in range(r.shape[0]): # loop over time stamps
#         Q[k,inds[0],inds[1]] = np.sum(r[k,inds_runoff[0],inds_runoff[1]]) # assign subglacial discharge

# # assign outside of convex hull values to nominal value
# for k in range(r.shape[0]):
#     Q[k,outside_indices[0],outside_indices[1]] = np.sum(r[k,:,:])/50

# # plot an example subglacial discharge field (typical values 100s-1000s m3/s in mid-summer)
# plt.pcolormesh(x,y,Q[7,:,:])
# plt.colorbar(label='subglacial discharge (m$^3$/s)')
# plt.axis('equal')
# plt.show()